<a href="https://colab.research.google.com/github/redfelix/CustomGLNotebook/blob/main/Trascrizione.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Installa l'ultima versione ufficiale di Whisper e assicurati che FFmpeg sia presente
!pip install -U openai-whisper
!sudo apt update && sudo apt install ffmpeg


Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:5 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
129 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt

In [1]:
import os
import subprocess
import torch
import whisper
from google.colab import files


# Modelli: tiny, base, small, medium, large, large-v3
modello = "large-v3"  # @param ["tiny", "base", "small", "medium", "large", "large-v3"]
lingua = "it"  # @param ["it", "en", "auto"]
elimina_file_sorgente = True  # @param {type:"boolean"}
scarica_txt = True  # @param {type:"boolean"}

# Tempo di inizio e fine (formato HH:MM:SS oppure MM:SS, lasciare 0 per tutto il file)
start_time = "00:00:12"  # @param {type:"string"}
end_time = "00:00:00"    # @param {type:"string"}

# Controllo se e' disponibile la GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo in uso: {device.upper()}")
if device == "cpu":
    print("Nota: nessuna GPU attiva. Per velocizzare seleziona una GPU da Runtime > Modifica tipo di runtime.")

print(f"Caricamento modello '{modello}' in corso...")
model = whisper.load_model(modello, device=device)
print("Modello caricato.")

#Caricamento file audio/video
print("\nCarica il file audio o video:")
uploaded = files.upload()

if not uploaded:
    print("Nessun file selezionato.")
else:
    for file_name in uploaded.keys():
        print(f"\n--- Inizio elaborazione: {file_name} ---")

        input_audio = file_name
        temp_clipped = None

        # Verifica se tagliare lo spezzone
        has_start = start_time.strip() not in ["", "0", "00:00:00", "00:00"]
        has_end = end_time.strip() not in ["", "0", "00:00:00", "00:00"]

        if has_start or has_end:
            temp_clipped = f"temp_cut_{os.path.splitext(file_name)[0]}.wav"
            cmd = ["ffmpeg", "-y", "-loglevel", "error"]

            if has_start:
                cmd.extend(["-ss", start_time])
            cmd.extend(["-i", file_name])
            if has_end:
                cmd.extend(["-to", end_time])

            # Salva l'audio temporaneo in wav a 16kHz
            cmd.extend(["-ar", "16000", "-ac", "1", "-c:a", "pcm_s16le", temp_clipped])

            print(f"Estrazione intervallo audio ({start_time} -> {end_time if has_end else 'fine'})...")
            subprocess.run(cmd, check=True)
            input_audio = temp_clipped

        #trascrizione del testo
        print("Trascrizione in corso con Whisper...")
        opzioni = {"verbose": False}
        if lingua != "auto":
            opzioni["language"] = lingua

        result = model.transcribe(input_audio, **opzioni)

        # Mostra il testo completo
        print("\n" + "=" * 50)
        print("TESTO TRASCRITTO:")
        print("=" * 50)
        print(result["text"].strip())
        print("=" * 50)

        # Dettaglio con i minuti dei segmenti
        print("\nSegmenti:")
        for seg in result["segments"]:
            m_in, s_in = divmod(seg["start"], 60)
            m_out, s_out = divmod(seg["end"], 60)
            print(f"[{int(m_in):02d}:{s_in:05.2f} -> {int(m_out):02d}:{s_out:05.2f}] {seg['text'].strip()}")

        #salvataggio e clean
        nome_base = os.path.splitext(file_name)[0]
        txt_path = f"{nome_base}_trascrizione.txt"

        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(result["text"].strip())

        if scarica_txt:
            print(f"\nDownload del file di testo: {txt_path}")
            files.download(txt_path)

        # Rimozione file temporanei e originali
        if temp_clipped and os.path.exists(temp_clipped):
            os.remove(temp_clipped)

        if elimina_file_sorgente and os.path.exists(file_name):
            os.remove(file_name)
            print(f"File sorgente {file_name} rimosso.")

        print(f"\n Trascrizione-elaborazione terminata per {file_name}.")

Dispositivo in uso: CUDA
Caricamento modello 'large-v3' in corso...
Modello caricato.

Carica il file audio o video:


Saving 2025-12-11 11-30-18.mp4 to 2025-12-11 11-30-18.mp4

--- Inizio elaborazione: 2025-12-11 11-30-18.mp4 ---
Estrazione intervallo audio (00:00:12 -> fine)...
Trascrizione in corso con Whisper...


100%|██████████| 551/551 [00:02<00:00, 205.53frames/s]


TESTO TRASCRITTO:
Diciamo io la potrei impostare così, ora vediamo pure loro, sono un po' eclettici.

Segmenti:
[00:00.00 -> 00:05.46] Diciamo io la potrei impostare così, ora vediamo pure loro, sono un po' eclettici.

Download del file di testo: 2025-12-11 11-30-18_trascrizione.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File sorgente 2025-12-11 11-30-18.mp4 rimosso.

 Trascrizione-elaborazione terminata per 2025-12-11 11-30-18.mp4.
